I need to attempt to solve the parse for the antialiased text.

The difficulty in our usecase is that the JPG wavelets are messing with the background color of the text, so naive matching performs worse than expected.

In [1]:
import dask
import numpy as np
import pandas as pd
from numpy import ndarray
from dask.array.image import imread

In [2]:
filename_pattern = r'/home/rainybyte/AllSkyImages/2019-09/*.JPG'
images = imread(filename_pattern)
images

dask.array<imread, shape=(12098, 480, 640, 3), dtype=uint8, chunksize=(1, 480, 640, 3), chunktype=numpy.ndarray>

Normally I would call `from allsky.classifiers import char_glyphs, char_width, classify_at_cursor, classify_fields_block` here
but we must not use the normal char_glyphs. I need to build new ones and see if the 3d (3-channel) glyphs will compare to the 3d (3-channel) images the same one the monocolor classifiers do.

In [3]:
from allsky.classifiers_antialiased import char_width, char_height, classify_at_cursor, classify_fields_block, \
    char_atlas, atlas_charcount, char_glyphs, initialize_antialiased_classifiers

In [4]:
from allsky.classifiers_antialiased import digit_templates, atlas_chars
digit_templates.shape

(11, 144)

In [5]:
np.stack([char_glyphs[ch] for ch in atlas_chars]).astype(np.float32).shape

(11, 8, 6, 3)

In [6]:
(np.stack([char_glyphs[ch] for ch in atlas_chars]).astype(np.float32) - np.stack([char_glyphs[ch] for ch in atlas_chars]).astype(np.float32).mean(axis=(1,2,3), keepdims=True)).reshape(11,-1).shape

(11, 144)

In [7]:
char_atlas

dask.array<getitem, shape=(8, 66, 3), dtype=uint8, chunksize=(8, 66, 3), chunktype=numpy.ndarray>

In [8]:
char_atlas.reshape(8, atlas_charcount, char_width, 3)

dask.array<reshape, shape=(8, 11, 6, 3), dtype=uint8, chunksize=(8, 11, 6, 3), chunktype=numpy.ndarray>

In [9]:
char_atlas.reshape(8, atlas_charcount, char_width, 3).transpose(1,0,2,3)

dask.array<transpose, shape=(11, 8, 6, 3), dtype=uint8, chunksize=(11, 8, 6, 3), chunktype=numpy.ndarray>

In [10]:
char_atlas.reshape(8, atlas_charcount, char_width, 3).transpose(1,0,2,3) # This is the minimal transpose so that I can index into `char_atlas` (I think)

dask.array<transpose, shape=(11, 8, 6, 3), dtype=uint8, chunksize=(11, 8, 6, 3), chunktype=numpy.ndarray>

In [11]:
char_atlas.reshape(8, atlas_charcount, char_width, 3).astype(np.float32).transpose(3, 1, 0, 2).compute() # this is if I wanted the transposed human-seeable shape

array([[[[  0., 137., 255., 255., 171.,  17.],
         [ 59., 245.,  37.,   1., 203., 103.],
         [147., 186.,  11.,   0., 117., 172.],
         ...,
         [120., 177.,  12.,   0., 125., 179.],
         [ 51., 239.,  40.,  12., 232., 110.],
         [  0., 137., 255., 251., 176.,  13.]],

        [[  0.,   0.,   5., 212.,  48.,   0.],
         [  0.,  10., 233., 255.,  48.,   0.],
         [  0., 204.,  50., 247.,  45.,   0.],
         ...,
         [  0.,   0.,  11., 235.,  48.,   0.],
         [  0.,   0.,  14., 241.,  43.,   0.],
         [  0.,   0.,   7., 233.,  46.,   0.]],

        [[  0., 201., 255., 255., 167.,   0.],
         [127., 202.,   0.,  58., 241.,  84.],
         [  0.,   1.,   0.,   0., 187.,  84.],
         ...,
         [  0.,  40., 227.,  54.,  15.,   0.],
         [ 37., 235.,  42.,  13.,   9.,   0.],
         [163., 255., 255., 255., 255.,  85.]],

        ...,

        [[  0., 122., 255., 255., 212.,  20.],
         [ 55., 210.,  34.,   0., 221., 139.]

In [12]:
best_char, best_score, scores = classify_at_cursor(
    image=images[0].astype(np.float32),
    x=5 + char_width * 3,
    y=6,
    atlas=char_glyphs,
    atlas_chars=char_glyphs.keys(),
    width=char_width,
    height=char_height,
)
best_char, best_score, scores

('9',
 0.9999988079071045,
 {'0': 0.7196726202964783,
  '1': -0.12133041024208069,
  '2': 0.47127991914749146,
  '3': 0.5679858922958374,
  '4': -0.11237775534391403,
  '5': 0.4873620569705963,
  '6': 0.4852602183818817,
  '7': 0.11555531620979309,
  '8': 0.5005043745040894,
  '9': 0.9999988079071045,
  's': 0.206564798951149})

2, 0, and 9 from "2019" parse fine. the number 1 seems to be an issue.

In [13]:
best_char, best_score, scores = classify_at_cursor(
    image=images[0].astype(np.float32),
    x=53,
    y=6,
    atlas=char_glyphs,
    atlas_chars=char_glyphs.keys(),
    width=char_width,
    height=char_height,
)
best_char, best_score, scores

('1',
 0.9937577247619629,
 {'0': -0.24705785512924194,
  '1': 0.9937577247619629,
  '2': -0.005819452926516533,
  '3': -0.0566893070936203,
  '4': 0.21375662088394165,
  '5': -0.07688598334789276,
  '6': -0.09145257622003555,
  '7': 0.11668256670236588,
  '8': -0.0668644905090332,
  '9': -0.12322845309972763,
  's': 0.24083071947097778})

But that "1" parsed fine? So it's just the one at x=18

In [14]:
best_char, best_score, scores = classify_at_cursor(
    image=images[0].astype(np.float32),
    x=5 + char_width * 2,
    y=6,
    atlas=char_glyphs,
    atlas_chars=char_glyphs.keys(),
    width=char_width,
    height=char_height,
)
best_char, best_score, scores

('1',
 0.9972930550575256,
 {'0': -0.2449895590543747,
  '1': 0.9972930550575256,
  '2': -0.009753053076565266,
  '3': -0.05688030272722244,
  '4': 0.20752164721488953,
  '5': -0.07449054718017578,
  '6': -0.09248360991477966,
  '7': 0.11858276277780533,
  '8': -0.06599762290716171,
  '9': -0.12710896134376526,
  's': 0.23141229152679443})

Fixed it! Error in the atlas, I had the "1" shifted one pixel over.

In [15]:
from allsky.classifiers_antialiased import (
    classify_date_string,
    classify_time_string,
    classify_exposure_string,
    classify_filename_string,
)

In [16]:
classify_date_string(images[0]), classify_time_string(images[0]), classify_exposure_string(images[0]), classify_filename_string(images[0])

('2019/08/31', '23:58:09', '49.4054', '001456621')

In [17]:
import os
from glob import glob

image_folders = sorted(glob("/home/rainybyte/AllSkyImages/20*/"))
len(image_folders), image_folders

(100,
 ['/home/rainybyte/AllSkyImages/2010-08/',
  '/home/rainybyte/AllSkyImages/2010-09/',
  '/home/rainybyte/AllSkyImages/2010-10/',
  '/home/rainybyte/AllSkyImages/2010-11/',
  '/home/rainybyte/AllSkyImages/2010-12/',
  '/home/rainybyte/AllSkyImages/2011-01/',
  '/home/rainybyte/AllSkyImages/2011-02/',
  '/home/rainybyte/AllSkyImages/2011-03/',
  '/home/rainybyte/AllSkyImages/2011-04/',
  '/home/rainybyte/AllSkyImages/2011-05/',
  '/home/rainybyte/AllSkyImages/2011-06/',
  '/home/rainybyte/AllSkyImages/2011-07/',
  '/home/rainybyte/AllSkyImages/2011-08/',
  '/home/rainybyte/AllSkyImages/2011-09/',
  '/home/rainybyte/AllSkyImages/2011-10/',
  '/home/rainybyte/AllSkyImages/2011-11/',
  '/home/rainybyte/AllSkyImages/2011-12/',
  '/home/rainybyte/AllSkyImages/2012-01/',
  '/home/rainybyte/AllSkyImages/2012-02/',
  '/home/rainybyte/AllSkyImages/2012-03/',
  '/home/rainybyte/AllSkyImages/2012-04/',
  '/home/rainybyte/AllSkyImages/2012-05/',
  '/home/rainybyte/AllSkyImages/2012-06/',
  '/h

In [18]:
folder_set = set([os.path.basename(dir) for dir in glob("/home/rainybyte/AllSkyImages/20*")])
file_set = set([os.path.basename(file)[:-8] for file in glob("../../data/*.parquet")])
missing_set = set([item for item in folder_set if item > "2016-02a"])
# missing_set = folder_set - file_set
missing_set = missing_set.difference({'2014-06'})
image_folders = [f"/home/rainybyte/AllSkyImages/{missing}/" for missing in sorted(list(missing_set))]
image_folders

['/home/rainybyte/AllSkyImages/2017-01/',
 '/home/rainybyte/AllSkyImages/2017-02/',
 '/home/rainybyte/AllSkyImages/2017-03/',
 '/home/rainybyte/AllSkyImages/2017-04/',
 '/home/rainybyte/AllSkyImages/2017-05/',
 '/home/rainybyte/AllSkyImages/2017-06/',
 '/home/rainybyte/AllSkyImages/2017-07/',
 '/home/rainybyte/AllSkyImages/2017-08/',
 '/home/rainybyte/AllSkyImages/2017-09/',
 '/home/rainybyte/AllSkyImages/2017-10/',
 '/home/rainybyte/AllSkyImages/2017-11/',
 '/home/rainybyte/AllSkyImages/2017-12/',
 '/home/rainybyte/AllSkyImages/2018-01/',
 '/home/rainybyte/AllSkyImages/2018-02/',
 '/home/rainybyte/AllSkyImages/2018-03/',
 '/home/rainybyte/AllSkyImages/2018-04/',
 '/home/rainybyte/AllSkyImages/2018-05/',
 '/home/rainybyte/AllSkyImages/2018-06/',
 '/home/rainybyte/AllSkyImages/2018-08/',
 '/home/rainybyte/AllSkyImages/2018-09/',
 '/home/rainybyte/AllSkyImages/2018-10/',
 '/home/rainybyte/AllSkyImages/2018-11/',
 '/home/rainybyte/AllSkyImages/2018-12/',
 '/home/rainybyte/AllSkyImages/201

In [19]:
from allsky.classifiers_antialiased import classify_fields_block, IMAGE_DIMENSIONS
IMAGE_DIMENSIONS

(480, 640, 3)

In [20]:

import os
from dask.array.image import imread

def process_allsky_image_folder_aa(directory: str, classify_fields_block):
    initialize_antialiased_classifiers()
    filename_pattern = f'{directory}*.JPG'
    images = imread(filename_pattern)
    all_batch = images.rechunk((32,480,640,3))
    df = all_batch.map_blocks(classify_fields_block, dtype=object, drop_axis=(1,2,3)).compute()
    df.to_parquet(f'../../data/{os.path.basename(directory[:-1])}.parquet')
    print("Done: ", directory)

# process_allsky_image_folder(image_folders[1])

tasks = [dask.delayed(process_allsky_image_folder_aa)(folder, classify_fields_block) for folder in sorted(image_folders)]
dask.compute(*tasks, scheduler='processes')
"All done!"

Done:  /home/rainybyte/AllSkyImages/2019-05/
Done:  /home/rainybyte/AllSkyImages/2017-09/
Done:  /home/rainybyte/AllSkyImages/2018-12/
Done:  /home/rainybyte/AllSkyImages/2018-11/
Done:  /home/rainybyte/AllSkyImages/2019-09/
Done:  /home/rainybyte/AllSkyImages/2020-02/
Done:  /home/rainybyte/AllSkyImages/2020-07/
Done:  /home/rainybyte/AllSkyImages/2018-09/
Done:  /home/rainybyte/AllSkyImages/2020-10/
Done:  /home/rainybyte/AllSkyImages/2018-02/
Done:  /home/rainybyte/AllSkyImages/2017-06/
Done:  /home/rainybyte/AllSkyImages/2017-04/
Done:  /home/rainybyte/AllSkyImages/2017-07/
Done:  /home/rainybyte/AllSkyImages/2018-08/
Done:  /home/rainybyte/AllSkyImages/2018-06/
Done:  /home/rainybyte/AllSkyImages/2017-03/
Done:  /home/rainybyte/AllSkyImages/2020-03/
Done:  /home/rainybyte/AllSkyImages/2019-06/
Done:  /home/rainybyte/AllSkyImages/2019-12/
Done:  /home/rainybyte/AllSkyImages/2020-11/
Done:  /home/rainybyte/AllSkyImages/2019-04/
Done:  /home/rainybyte/AllSkyImages/2019-11/
Done:  /ho

'All done!'